In [0]:
fetch_date = dbutils.widgets.get('fetch_date')
target_table = dbutils.widgets.get('target_table')
lastpaymnet_table = dbutils.widgets.get('lastpaymnet_table')

In [0]:
display(
spark.sql(f"""  
-- Step 1: Update Source System from HCHB to CubHub for specific invoice patterns
UPDATE {target_table}
SET Source_System = 'CubHub'
WHERE reporting_date = CAST('{fetch_date}' AS DATE)
AND Source_System = 'HCHB'
AND (
    Invoice_Number RLIKE '[A-Z][A-Z]' 
    OR Invoice_Number RLIKE '[A-Z][A-Z]S'
);
""")
)

In [0]:
display(
spark.sql(f"""  
-- Step 2: Update Last Payment Date from LastPayment table for CubHub records
MERGE INTO {target_table} AS target
USING (
    SELECT 
        ma.Invoice_Number,
        lp.entry_date AS Last_Payment_Date
    FROM {target_table} ma
    INNER JOIN {lastpaymnet_table} lp
        ON lp.claim_number = ma.Invoice_Number
    WHERE ma.reporting_date = CAST('{fetch_date}' AS DATE)
    AND ma.Source_System = 'CubHub'
) AS source
ON target.Invoice_Number = source.Invoice_Number
   AND target.reporting_date = CAST('{fetch_date}' AS DATE)
WHEN MATCHED THEN
    UPDATE SET target.Last_Payment_Date = source.Last_Payment_Date;
""")
)

In [0]:
display(
spark.sql(f"""  
-- Step 3: Clear NULL Last Payment Date values
UPDATE {target_table}
SET Last_Payment_Date = ' '
WHERE Last_Payment_Date IS NULL
AND reporting_date = CAST('{fetch_date}' AS DATE);
""")
)